# Level 4: Symbolically Supervised Neural Intent Classifier

**End-to-end walkthrough: data audit → train → infer → evaluate**

This notebook demonstrates the core Level 4 claim:
> Ontology-derived symbolic constraints can be encoded as a differentiable training-time loss term,
> allowing the neural model to learn constraint satisfaction without any symbolic post-processing at inference.

**Architecture summary:**
```
utterance → frozen all-MiniLM-L6-v2 encoder (384-dim)
          → shared Linear(384→256) + ReLU + Dropout
          → intent_head (→4) | entity_head (→7) | domain_head (→1)
          trained with: L = L_intent + α·L_entity + β·L_domain + λ·L_constraint
```

**Kautz typology position:** Type 4 — symbolic knowledge compiled into weights at training time, pure neural inference.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import torch
import pandas as pd

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Project root: {PROJECT_ROOT}")

## 1. Dataset audit

Verify the dataset is logically consistent before training:
- All SRE intents must be `domain_valid=True`
- `out_of_scope` must be `domain_valid=False`  
- No constraint violations in ground-truth labels

In [ ]:
df = pd.read_csv(PROJECT_ROOT / 'level4' / 'data' / 'labeled_clean.csv')
print(f"Total rows: {len(df)}")
print()
print("Intent x domain_valid cross-tab:")
print(pd.crosstab(df['intent'], df['domain_valid']))

In [ ]:
print("Intent x entity_type cross-tab:")
print(pd.crosstab(df['intent'], df['entity_type']))

In [ ]:
# Check constraint violations in labels
rules_path = PROJECT_ROOT / 'level4' / 'ontology' / 'constraint_rules.json'
with open(rules_path) as f:
    rules_cfg = json.load(f)

violations = 0
for rule in rules_cfg['disallowed_intent_entity_pairs']:
    count = ((df['intent'] == rule['intent']) & (df['entity_type'] == rule['entity_type'])).sum()
    if count > 0:
        print(f"  VIOLATION: {rule['intent']} + {rule['entity_type']} = {count} rows")
        violations += count

if violations == 0:
    print("Labels are constraint-clean (0 violations).")

## 2. Constraint rules

The 10 ontology-derived disallowed pairs that become the training-time loss signal.

In [ ]:
print(f"{'Tier':<12} {'Intent':<20} {'Entity':<18} {'Weight'}")
print("-" * 62)
for rule in rules_cfg['disallowed_intent_entity_pairs']:
    # Infer tier from weight
    w = rule['penalty_weight']
    tier = 'TYPE_A' if w == 1.0 else ('TYPE_B' if w == 0.75 else 'TYPE_C')
    print(f"{tier:<12} {rule['intent']:<20} {rule['entity_type']:<18} {w}")

## 3. Model architecture

In [ ]:
from level4.model.neural_intent_model import Level4IntentModel

model = Level4IntentModel()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}  (encoder frozen)")
print()
print(model)

## 4. Loss function

The constraint loss is differentiable: it operates on softmax probabilities so gradients flow back through both the intent and entity classification heads.

$$\mathcal{L}_\text{constraint} = \sum_{(i,j) \in \mathcal{D}} w_{ij} \cdot \mathbb{E}[p_\text{intent}^{(i)} \cdot p_\text{entity}^{(j)}]$$

In [ ]:
from level4.model.losses import Level4Loss

loss_fn = Level4Loss(lam=0.5)
cl = loss_fn.constraint_loss_fn

from level4.model.dataset import IDX_TO_INTENT, IDX_TO_ENTITY_TYPE
print(f"Loaded {len(cl.intent_idxs)} constraint pairs into SymbolicConstraintLoss:")
for i in range(len(cl.intent_idxs)):
    print(f"  {IDX_TO_INTENT[cl.intent_idxs[i].item()]:<20} + {IDX_TO_ENTITY_TYPE[cl.entity_idxs[i].item()]:<15}  w={cl.weights[i].item()}")

## 5. Training a model

Run `train.py` from the command line:
```bash
# Baseline (no symbolic loss)
python -m level4.train --lam 0.0 --epochs 20 --run-name lam0_0

# Symbolically supervised
python -m level4.train --lam 0.5 --epochs 20 --run-name lam0_5
```

Or demonstrate a single-batch forward+backward pass inline:

In [ ]:
from level4.model.dataset import IntentDataset
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        'utterances': [b['utterance'] for b in batch],
        'intent_idx':   torch.stack([b['intent_idx']   for b in batch]),
        'entity_idx':   torch.stack([b['entity_idx']   for b in batch]),
        'domain_valid': torch.stack([b['domain_valid'] for b in batch]),
    }

device = torch.device('cpu')
ds = IntentDataset(str(PROJECT_ROOT / 'level4' / 'data' / 'train.csv'))
loader = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
batch = next(iter(loader))

model.train()
loss_fn = Level4Loss(lam=0.5)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=3e-4
)

optimizer.zero_grad()
out = model.forward(batch['utterances'], device)
loss_dict = loss_fn(
    out['intent_logits'], out['entity_logits'], out['domain_logits'],
    batch['intent_idx'].to(device),
    batch['entity_idx'].to(device),
    batch['domain_valid'].to(device),
)
loss_dict['loss'].backward()
optimizer.step()

print("Single batch forward+backward pass:")
for k, v in loss_dict.items():
    print(f"  {k:<20}: {v:.4f}" if isinstance(v, float) else f"  {k:<20}: {v.item():.4f}")

## 6. Inference — clean neural prediction

Load a trained checkpoint and run `model.predict()`.  
No symbolic post-processing — the model output is the final answer.

In [ ]:
CHECKPOINT = PROJECT_ROOT / 'level4' / 'saved_models' / 'lam1_0' / 'best_model.pt'

if not CHECKPOINT.exists():
    print(f"Checkpoint not found: {CHECKPOINT}")
    print("Run: python -m level4.train --lam 1.0 --epochs 20 --run-name lam1_0")
else:
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    inf_model = Level4IntentModel()
    inf_model.load_state_dict(ckpt['state_dict'])
    inf_model.eval()
    print(f"Loaded: lam={ckpt['lam']}, epoch={ckpt['epoch']}, val_intent_acc={ckpt['val_intent_acc']:.4f}")

In [ ]:
test_utterances = [
    "restart the payment-service deployment",
    "show me the error rate for the checkout API over the last hour",
    "summarize all incidents from last week",
    "what is the capital of France",
    "scale down the batch job for nightly processing",
]

if CHECKPOINT.exists():
    preds = inf_model.predict(test_utterances)
    for p in preds:
        print(f"  [{p['intent']:<14}] [{p['entity_type']:<14}] domain={p['domain_valid']}  | {p['utterance']}")

## 7. Evaluation — constraint violation metrics

Load all saved experiment results and display the comparison table.

In [ ]:
SAVED = PROJECT_ROOT / 'level4' / 'saved_models'

runs = [
    ('level3_5_baseline', 'Level 3.5 (runtime symbolic)'),
    ('lam0_0',  'Level 4 λ=0.0 (baseline neural)'),
    ('lam0_1',  'Level 4 λ=0.1'),
    ('lam0_25', 'Level 4 λ=0.25'),
    ('lam0_5',  'Level 4 λ=0.5'),
    ('lam1_0',  'Level 4 λ=1.0  ← recommended'),
    ('lam2_0',  'Level 4 λ=2.0'),
]

rows = []
for run_dir, label in runs:
    p = SAVED / run_dir / 'evaluation_metrics.json'
    if p.exists():
        with open(p) as f:
            m = json.load(f)
        rows.append({'Run': label, **{k: m.get(k, '—') for k in [
            'intent_acc', 'entity_acc', 'domain_acc',
            'overall_violation_rate', 'type_a_false_rejection',
            'type_b_false_execution', 'type_c_ungrounded_sre'
        ]}})

results_df = pd.DataFrame(rows).set_index('Run')
results_df.columns = ['Intent Acc', 'Entity Acc', 'Domain Acc',
                       'Viol Rate', 'TYPE_A FR', 'TYPE_B FE', 'TYPE_C US']
results_df

In [ ]:
import matplotlib.pyplot as plt

ablation_runs = [
    ('lam0_0', 0.0), ('lam0_1', 0.1), ('lam0_25', 0.25),
    ('lam0_5', 0.5), ('lam1_0', 1.0), ('lam2_0', 2.0)
]

lam_vals, intent_accs, viol_rates, type_c_rates = [], [], [], []
for run_dir, lam in ablation_runs:
    p = SAVED / run_dir / 'evaluation_metrics.json'
    if p.exists():
        with open(p) as f:
            m = json.load(f)
        lam_vals.append(lam)
        intent_accs.append(m['intent_acc'])
        viol_rates.append(m['overall_violation_rate'])
        type_c_rates.append(m['type_c_ungrounded_sre'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(lam_vals, intent_accs, 'o-b', label='Intent Acc')
ax1.plot(lam_vals, viol_rates,  's-r', label='Violation Rate')
ax1.plot(lam_vals, type_c_rates,'D-g', label='TYPE_C Rate')
ax1.set_xlabel('λ (constraint loss weight)')
ax1.set_ylabel('Rate')
ax1.set_title('λ Ablation: Accuracy vs Violation Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='recommended λ=1.0')

# Bar chart: violation rate by system
system_labels = ['Level 3.5\n(runtime)', 'L4 λ=0\n(baseline)', 'L4 λ=0.5', 'L4 λ=1.0', 'L4 λ=2.0']
system_viol   = []
for rd in ['level3_5_baseline', 'lam0_0', 'lam0_5', 'lam1_0', 'lam2_0']:
    p = SAVED / rd / 'evaluation_metrics.json'
    if p.exists():
        with open(p) as f:
            system_viol.append(json.load(f)['overall_violation_rate'])

colors = ['#d62728'] + ['#aec7e8'] * 2 + ['#1f77b4'] + ['#aec7e8']
ax2.bar(system_labels, system_viol, color=colors[:len(system_viol)])
ax2.set_ylabel('Constraint Violation Rate')
ax2.set_title('Violation Rate by System')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'level4' / 'saved_models' / 'ablation_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved to level4/saved_models/ablation_plot.png")

## 8. Summary

**What Level 4 achieves over Level 3.5:**

| Metric | Level 3.5 | Level 4 λ=1.0 |
|---|---|---|
| Intent accuracy | 0.931 | **0.967** |
| Entity accuracy | 0.372 | **0.682** |
| Constraint violation rate | 0.643 | **0.036** |
| Runtime symbolic components | Yes | **No** |

**Key conclusions:**

1. Training-time symbolic constraint loss reduces violation rates by **~95%** compared to Level 3.5's runtime grounding.
2. TYPE_B false execution violations are eliminated at λ≥0.1 — the model internalises the `execution+metric` and `execution+incident` constraints.
3. Intent accuracy is **higher** than Level 3.5 (0.967 vs 0.931) while using no runtime symbolic components.
4. λ=1.0 is the recommended operating point: best violation reduction without meaningful accuracy cost.
5. λ=2.0 halves violations further (2.1%) with only 0.9% accuracy cost — the constraint signal dominates but does not collapse classification.

**This is the Level 4 claim:** ontology constraints can be compiled into neural weights, making the model itself constraint-aware at inference time.